# IEEE Clean Data

Prepare the IEEE fraud dataset into the same raw tabular format expected by `run_experiment.py`:

- `TransactionDT` -> `dt`
- `isFraud` -> `label`
- selected columns renamed to `feature_i`
- all feature columns numeric so the current preprocessing pipeline can run end-to-end
- export a sidecar metadata JSON for feature provenance and shared-feature prioritization

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import os
import zipfile

import numpy as np
import pandas as pd

DATA_ROOT = Path(os.environ.get('EVO_CFD_DATA_ROOT', '/data'))
INPUT_DIR = DATA_ROOT / 'ieee-fraud-detection'
IEEE_ZIP = Path(os.environ.get('IEEE_FRAUD_ZIP', str(DATA_ROOT / 'ieee-fraud-detection.zip')))
OUTPUT_CSV = DATA_ROOT / 'fraud_corp_ieee_output_2024.csv'
OUTPUT_META = DATA_ROOT / 'fraud_corp_ieee_output_2024.feature_metadata.json'
ADOPTED_FEATURE_NUM = 180
MISSING_SENTINEL = -1.0

In [ ]:

def ensure_ieee_raw_files():
    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    transaction_candidates = ['train_transaction.csv', 'train_transactions.csv']
    identity_candidates = ['train_identity.csv', 'train_identities.csv']
    has_transaction = any((INPUT_DIR / name).exists() for name in transaction_candidates)
    has_identity = any((INPUT_DIR / name).exists() for name in identity_candidates)
    if has_transaction and has_identity:
        return
    if not IEEE_ZIP.exists():
        raise FileNotFoundError(
            f'Missing IEEE raw files under {INPUT_DIR}. Set IEEE_FRAUD_ZIP or place the zip at {IEEE_ZIP}.'
        )
    with zipfile.ZipFile(IEEE_ZIP) as archive:
        archive.extractall(INPUT_DIR)


def first_existing_file(directory: Path, names):
    for name in names:
        path = directory / name
        if path.exists():
            return path
    expected = ', '.join(names)
    raise FileNotFoundError(f'Expected one of [{expected}] under {directory}')


def normalize_identity_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns=lambda col: col.replace('-', '_'))


def classify_feature_source(column: str) -> str:
    if column.startswith('id_') or column in {'DeviceType', 'DeviceInfo'}:
        return 'identity_table'
    if column.startswith(('card', 'addr', 'dist')) or column in {'P_emaildomain', 'R_emaildomain'}:
        return 'entity_descriptor'
    if column == 'ProductCD':
        return 'transaction_context'
    if column.startswith('V'):
        return 'engineered_V'
    if column.startswith('C'):
        return 'count_C'
    if column.startswith('D'):
        return 'timedelta_D'
    if column.startswith('M'):
        return 'match_M'
    return 'transaction'


def is_identity_like(column: str) -> bool:
    source = classify_feature_source(column)
    return source in {
        'identity_table',
        'entity_descriptor',
        'transaction_context',
        'timedelta_D',
        'match_M',
    }


def encode_series_to_numeric(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype('float32')

    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors='coerce')
        numeric = numeric.replace([np.inf, -np.inf], np.nan)
        return numeric.astype('float32')

    as_str = series.astype('string')
    missing_mask = as_str.isna()
    categories = sorted(list(as_str[~missing_mask].unique()))
    mapping = {value: idx for idx, value in enumerate(categories)}
    encoded = as_str.map(mapping).astype('float32')
    encoded[missing_mask] = MISSING_SENTINEL
    return encoded


def build_ieee_training_frame(adopted_feature_num: int = ADOPTED_FEATURE_NUM):
    ensure_ieee_raw_files()
    transaction_path = first_existing_file(INPUT_DIR, ['train_transaction.csv', 'train_transactions.csv'])
    identity_path = first_existing_file(INPUT_DIR, ['train_identity.csv', 'train_identities.csv'])
    train_transaction = pd.read_csv(transaction_path)
    train_identity = normalize_identity_columns(pd.read_csv(identity_path))

    merged = train_transaction.merge(train_identity, on='TransactionID', how='left')

    feature_candidates = [
        column for column in merged.columns
        if column not in {'TransactionID', 'isFraud', 'TransactionDT'}
    ]

    null_counts = merged[feature_candidates].isna().sum().sort_values(kind='stable')
    selected_original_columns = null_counts.head(adopted_feature_num).index.tolist()

    base_output = pd.DataFrame({
        'dt': merged['TransactionDT'].astype('int64'),
        'label': merged['isFraud'].astype('int8'),
    })

    encoded_feature_columns = {}
    feature_records = []
    identity_like_features = []

    for feature_idx, original_column in enumerate(selected_original_columns):
        feature_name = f'feature_{feature_idx}'
        encoded = encode_series_to_numeric(merged[original_column])
        encoded_feature_columns[feature_name] = encoded

        source_group = classify_feature_source(original_column)
        identity_like = is_identity_like(original_column)
        if identity_like:
            identity_like_features.append(feature_name)

        feature_records.append({
            'feature_name': feature_name,
            'original_name': original_column,
            'source_group': source_group,
            'is_identity_like': identity_like,
            'null_count': int(null_counts[original_column]),
            'null_ratio': float(null_counts[original_column] / len(merged)),
            'encoded_dtype': str(encoded.dtype),
        })

    feature_frame = pd.DataFrame(encoded_feature_columns)
    output = pd.concat([base_output, feature_frame], axis=1)

    metadata = {
        'dataset': 'ieee',
        'row_count': int(len(output)),
        'adopted_feature_num': int(adopted_feature_num),
        'output_file': str(OUTPUT_CSV),
        'feature_metadata_file': str(OUTPUT_META),
        'dt_column': 'TransactionDT',
        'label_column': 'isFraud',
        'selected_original_columns': selected_original_columns,
        'identity_like_features': identity_like_features,
        'feature_map': feature_records,
    }

    return output, metadata


def export_ieee_training_frame(adopted_feature_num: int = ADOPTED_FEATURE_NUM):
    frame, metadata = build_ieee_training_frame(adopted_feature_num=adopted_feature_num)
    frame.to_csv(OUTPUT_CSV, index=False)
    with open(OUTPUT_META, 'w') as handle:
        json.dump(metadata, handle, indent=2)
    return frame, metadata

In [ ]:
clean_df, clean_meta = export_ieee_training_frame()
print({
    'rows': len(clean_df),
    'cols': len(clean_df.columns),
    'feature_cols': len([c for c in clean_df.columns if c.startswith('feature_')]),
    'identity_like_features': len(clean_meta['identity_like_features']),
    'output_csv': str(OUTPUT_CSV),
    'output_meta': str(OUTPUT_META),
})
clean_df.head()